# Atividade 2 — Camada Gold e Desafio de Analytics

Este notebook constrói dimensões, tabelas-ponte e uma tabela de contexto para IA a partir da camada Silver. No final, apresenta as seis respostas pedidas no enunciado.

## 1. Preparação, funções de auxiliares e verificação de Silver

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DecimalType

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

def salvar_gold(df, tabela):
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(tabela))

def chave_substituta(df, colunas_chave, nome_sk):
    campos = [F.col(c).alias(f"k{i}") for i, c in enumerate(colunas_chave)]
    json_chave = F.to_json(F.struct(*campos), {"ignoreNullFields": "false"})
    sk = F.conv(F.substring(F.sha2(json_chave, 256), 1, 15), 16, 10).cast("bigint")
    return df.withColumn(nome_sk, sk).select(nome_sk, *df.columns)

def exigir_unicidade(df, coluna, rotulo):
    duplicados = (df.groupBy(coluna).count()
                    .filter(F.col("count") > 1)
                    .limit(1).count())
    if duplicados:
        raise ValueError(f"Chave duplicada em {rotulo}.{coluna}")

def mostrar_contagem(df, rotulo):
    print(f"{rotulo}: {df.count()} linhas")


## 2. Dimensões de filme, gênero, pessoas e produtoras


In [0]:
info_silver = spark.table("silver.tb_info_filmes")

dim_movies_base = (info_silver
    .filter(F.col("id_filme").isNotNull())
    .select(
        F.col("id_filme").cast("string").alias("id_filme"),
        F.col("titulo").cast("string").alias("titulo"),
        F.col("data_lancamento").cast("date").alias("data_lancamento"),
        F.col("ano_lancamento").cast("int").alias("ano_lancamento"),
        F.col("duracao_minutos").cast("int").alias("duracao_minutos"),
        F.col("idioma_original").cast("string").alias("idioma_original"),
        F.col("status_filme").cast("string").alias("status_filme"),
        F.col("sinopse").cast("string").alias("sinopse")
    )
    .dropDuplicates(["id_filme"]))
dim_movies = chave_substituta(dim_movies_base, ["id_filme"], "sk_movie_id")
dim_movies = dim_movies.select(
    F.col("sk_movie_id").cast("bigint"), "id_filme", "titulo", "data_lancamento",
    "ano_lancamento", "duracao_minutos", "idioma_original", "status_filme", "sinopse"
)
exigir_unicidade(dim_movies, "sk_movie_id", "gold.dim_movies")
exigir_unicidade(dim_movies, "id_filme", "gold.dim_movies")
salvar_gold(dim_movies, "gold.dim_movies")

generos_silver = spark.table("silver.tb_generos")
dim_genres_base = (generos_silver
    .select(F.trim(F.col("genero")).alias("nome_genero"))
    .filter(F.col("nome_genero").isNotNull() & (F.col("nome_genero") != ""))
    .dropDuplicates(["nome_genero"]))
dim_genres = chave_substituta(dim_genres_base, ["nome_genero"], "sk_genre_id")
dim_genres = dim_genres.select(F.col("sk_genre_id").cast("bigint"), "nome_genero")
exigir_unicidade(dim_genres, "sk_genre_id", "gold.dim_genres")
salvar_gold(dim_genres, "gold.dim_genres")

pessoas_silver = spark.table("silver.tb_pessoas_empresas")
dim_people_base = (pessoas_silver
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.trim(F.col("nome_entidade")).alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .filter(F.col("nome_pessoa").isNotNull() & (F.col("nome_pessoa") != ""))
    .dropDuplicates(["nome_pessoa", "tipo_pessoa"]))
dim_people = chave_substituta(dim_people_base, ["nome_pessoa", "tipo_pessoa"], "sk_person_id")
dim_people = dim_people.select(F.col("sk_person_id").cast("bigint"), "nome_pessoa", "tipo_pessoa")
exigir_unicidade(dim_people, "sk_person_id", "gold.dim_people")
salvar_gold(dim_people, "gold.dim_people")

dim_companies_base = (pessoas_silver
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.trim(F.col("nome_entidade")).alias("nome_produtora"))
    .filter(F.col("nome_produtora").isNotNull() & (F.col("nome_produtora") != ""))
    .dropDuplicates(["nome_produtora"]))
dim_companies = chave_substituta(dim_companies_base, ["nome_produtora"], "sk_company_id")
dim_companies = dim_companies.select(F.col("sk_company_id").cast("bigint"), "nome_produtora")
exigir_unicidade(dim_companies, "sk_company_id", "gold.dim_companies")
salvar_gold(dim_companies, "gold.dim_companies")


## 3. Resumo das avaliações por filme

In [0]:
avaliacoes_silver = spark.table("silver.tb_avaliacoes_usuarios")
resumo_avaliacoes = (avaliacoes_silver
    .groupBy("id_filme")
    .agg(
        F.count(F.lit(1)).cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
    ))
resumo_com_fk = (resumo_avaliacoes
    .join(dim_movies.select("id_filme", "sk_movie_id"), "id_filme", "inner")
    .select("sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios"))
dim_reviews = chave_substituta(resumo_com_fk, ["sk_movie_id"], "sk_review_id")
dim_reviews = dim_reviews.select(
    F.col("sk_review_id").cast("bigint"), F.col("sk_movie_id").cast("bigint"),
    F.col("qtd_avaliacoes_usuarios").cast("int"),
    F.col("nota_media_usuarios").cast("double")
)
exigir_unicidade(dim_reviews, "sk_review_id", "gold.dim_reviews")
exigir_unicidade(dim_reviews, "sk_movie_id", "gold.dim_reviews")
salvar_gold(dim_reviews, "gold.dim_reviews")
display(dim_reviews.limit(10))


## 4. Uma linha por filme lançado

In [0]:
filmes_lancados = (dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme"))
financeiro_silver = spark.table("silver.tb_financeiro_filmes")
metricas_silver = spark.table("silver.tb_metricas_engajamento")
exigir_unicidade(financeiro_silver, "id_filme", "silver.tb_financeiro_filmes")
exigir_unicidade(metricas_silver, "id_filme", "silver.tb_metricas_engajamento")

fato_join = (filmes_lancados
    .join(financeiro_silver, "id_filme", "left")
    .join(metricas_silver, "id_filme", "left"))
fact_movies_performance = fato_join.select(
    F.col("sk_movie_id").cast("bigint").alias("sk_movie_id"),
    F.col("orcamento_usd").cast(DecimalType(18, 2)).alias("orcamento_usd"),
    F.col("receita_usd").cast(DecimalType(18, 2)).alias("receita_usd"),
    F.col("lucro_usd").cast(DecimalType(18, 2)).alias("lucro_usd"),
    F.col("orcamento_brl").cast(DecimalType(18, 2)).alias("orcamento_brl"),
    F.col("receita_brl").cast(DecimalType(18, 2)).alias("receita_brl"),
    F.col("lucro_brl").cast(DecimalType(18, 2)).alias("lucro_brl"),
    F.col("popularidade").cast("double").alias("popularidade"),
    F.col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
    F.col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
    F.col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
    F.col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb")
)
exigir_unicidade(fact_movies_performance, "sk_movie_id", "gold.fact_movies_performance")
if fact_movies_performance.count() != filmes_lancados.count():
    raise ValueError("A fato não preservou exatamente uma linha por filme lançado.")
salvar_gold(fact_movies_performance, "gold.fact_movies_performance")
display(fact_movies_performance.limit(10))


## 5. Tabelas-ponte para relações muitos-para-muitos

In [0]:
# Filme <-> gênero
pares_genero = (generos_silver
    .select("id_filme", F.col("genero").alias("nome_genero"))
    .dropDuplicates(["id_filme", "nome_genero"]))
bridge_movie_genre = (pares_genero
    .join(dim_movies.select("id_filme", "sk_movie_id"), "id_filme", "inner")
    .join(dim_genres, "nome_genero", "inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_genre_id").cast("bigint"))
    .dropDuplicates())
salvar_gold(bridge_movie_genre, "gold.bridge_movie_genre")

# Filme <-> pessoa
pares_pessoa = (pessoas_silver
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        "id_filme", F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .dropDuplicates(["id_filme", "nome_pessoa", "tipo_pessoa"]))
bridge_movie_person = (pares_pessoa
    .join(dim_movies.select("id_filme", "sk_movie_id"), "id_filme", "inner")
    .join(dim_people, ["nome_pessoa", "tipo_pessoa"], "inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_person_id").cast("bigint"))
    .dropDuplicates())
salvar_gold(bridge_movie_person, "gold.bridge_movie_person")

# Filme <-> produtora
pares_produtora = (pessoas_silver
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("id_filme", F.col("nome_entidade").alias("nome_produtora"))
    .dropDuplicates(["id_filme", "nome_produtora"]))
bridge_movie_company = (pares_produtora
    .join(dim_movies.select("id_filme", "sk_movie_id"), "id_filme", "inner")
    .join(dim_companies, "nome_produtora", "inner")
    .select(F.col("sk_movie_id").cast("bigint"), F.col("sk_company_id").cast("bigint"))
    .dropDuplicates())
salvar_gold(bridge_movie_company, "gold.bridge_movie_company")

display(bridge_movie_genre.limit(10))
display(bridge_movie_person.limit(10))
display(bridge_movie_company.limit(10))


## 6. Tabela de contexto para IA

In [0]:
generos_por_filme = (bridge_movie_genre
    .join(dim_genres, "sk_genre_id")
    .groupBy("sk_movie_id")
    .agg(F.sort_array(F.collect_set("nome_genero")).alias("generos")))

pessoas_por_filme = (bridge_movie_person
    .join(dim_people, "sk_person_id")
    .groupBy("sk_movie_id")
    .agg(
        F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Ator", F.col("nome_pessoa")))).alias("atores"),
        F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Diretor", F.col("nome_pessoa")))).alias("diretores"),
        F.sort_array(F.collect_set(F.when(F.col("tipo_pessoa") == "Roteirista", F.col("nome_pessoa")))).alias("roteiristas")
    ))

produtoras_por_filme = (bridge_movie_company
    .join(dim_companies, "sk_company_id")
    .groupBy("sk_movie_id")
    .agg(F.sort_array(F.collect_set("nome_produtora")).alias("produtoras")))

contexto = (dim_movies
    .join(generos_por_filme, "sk_movie_id", "left")
    .join(pessoas_por_filme, "sk_movie_id", "left")
    .join(produtoras_por_filme, "sk_movie_id", "left")
    .join(fact_movies_performance, "sk_movie_id", "left")
    .join(dim_reviews, "sk_movie_id", "left"))
segmentos_contexto = [
    F.concat(F.lit("Título: "), F.coalesce(F.col("titulo"), F.lit("Não informado"))),
    F.concat(F.lit("Lançamento: "), F.coalesce(F.col("data_lancamento").cast("string"), F.lit("Não informado"))),
    F.concat(F.lit("Sinopse: "), F.coalesce(F.col("sinopse"), F.lit("Não informada"))),
    F.concat(F.lit("Gêneros: "), F.coalesce(F.concat_ws(", ", F.col("generos")), F.lit("Não informado"))),
    F.concat(F.lit("Elenco: "), F.coalesce(F.concat_ws(", ", F.col("atores")), F.lit("Não informado"))),
    F.concat(F.lit("Direção: "), F.coalesce(F.concat_ws(", ", F.col("diretores")), F.lit("Não informada"))),
    F.concat(F.lit("Roteiro: "), F.coalesce(F.concat_ws(", ", F.col("roteiristas")), F.lit("Não informado"))),
    F.concat(F.lit("Produtoras: "), F.coalesce(F.concat_ws(", ", F.col("produtoras")), F.lit("Não informadas")))
]
contexto = contexto.withColumn("contexto_filme", F.concat_ws(" | ", *segmentos_contexto))
salvar_gold(contexto, "gold.context_movies_ai")
display(contexto.select("sk_movie_id", "titulo", "generos", "atores", "contexto_filme").limit(10))


## 7. Desafio de Analytics

### 1. Receita total em reais


In [0]:
display(fact_movies_performance.agg(
    F.sum("receita_brl").cast(DecimalType(38, 2)).alias("receita_total_brl")
))


### 2. Cinco filmes com maior popularidade


In [0]:
display(fact_movies_performance
    .join(dim_movies.select("sk_movie_id", "titulo"), "sk_movie_id", "inner")
    .filter(F.col("popularidade").isNotNull())
    .select("titulo", "popularidade")
    .orderBy(F.col("popularidade").desc(), F.col("titulo").asc())
    .limit(5))


### 3. Quantidade de filmes por gênero


In [0]:
display(bridge_movie_genre
    .join(dim_genres, "sk_genre_id", "inner")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc(), F.col("nome_genero").asc()))


### 4. Dez filmes de maior receita

In [0]:
janela_rank_receita = Window.orderBy(F.col("receita_usd").desc())
rank_receitas = (fact_movies_performance
    .filter(F.col("receita_usd").isNotNull())
    .join(dim_movies.select("sk_movie_id", "titulo"), "sk_movie_id", "inner")
    .withColumn("posicao_rank", F.rank().over(janela_rank_receita))
    .filter(F.col("posicao_rank") <= 10)
    .select("posicao_rank", "titulo", "receita_usd", "receita_brl")
    .orderBy("posicao_rank", F.col("titulo").asc()))
display(rank_receitas)


### Referência para as perguntas 5 e 6


In [0]:
data_referencia = (dim_movies
    .filter((F.col("status_filme") == "Lançado") & (F.col("data_lancamento") <= F.current_date()))
    .agg(F.max("data_lancamento").alias("data_referencia"))
    .first()["data_referencia"])
if data_referencia is None:
    raise ValueError("Não há data de lançamento válida de filme lançado para definir o período analítico.")

def filmes_no_periodo(anos):
    data_limite = F.add_months(F.lit(data_referencia), -12 * anos)
    return dim_movies.filter(
        (F.col("status_filme") == "Lançado")
        & (F.col("data_lancamento") >= data_limite)
        & (F.col("data_lancamento") <= F.lit(data_referencia))
    )


### 5. Ator com mais participações em filmes lançados nos últimos dois anos

In [0]:
participacoes_atores = (filmes_no_periodo(2)
    .select("sk_movie_id")
    .join(bridge_movie_person, "sk_movie_id", "inner")
    .join(dim_people.filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id", "inner")
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes")))
janela_atores = Window.orderBy(F.col("qtd_filmes").desc())
resposta_ator = (participacoes_atores
    .withColumn("posicao", F.rank().over(janela_atores))
    .filter(F.col("posicao") == 1)
    .orderBy("nome_pessoa"))
display(resposta_ator)


### 6. Produtora com maior lucro nos últimos cinco anos

In [0]:
lucro_por_produtora = (filmes_no_periodo(5)
    .select("sk_movie_id")
    .join(fact_movies_performance, "sk_movie_id", "inner")
    .filter(F.col("lucro_usd").isNotNull())
    .join(bridge_movie_company, "sk_movie_id", "inner")
    .join(dim_companies, "sk_company_id", "inner")
    .groupBy("nome_produtora")
    .agg(
        F.sum("lucro_usd").cast(DecimalType(38, 2)).alias("lucro_total_usd"),
        F.sum("lucro_brl").cast(DecimalType(38, 2)).alias("lucro_total_brl")
    ))
janela_produtoras = Window.orderBy(F.col("lucro_total_usd").desc_nulls_last())
resposta_produtora = (lucro_por_produtora
    .withColumn("posicao", F.rank().over(janela_produtoras))
    .filter(F.col("posicao") == 1)
    .orderBy("nome_produtora"))
display(resposta_produtora)


## 8. Verificação de tabelas e chaves


In [0]:
tabelas_gold = [
    "dim_movies", "dim_genres", "dim_people", "dim_companies", "dim_reviews",
    "fact_movies_performance", "bridge_movie_genre", "bridge_movie_person",
    "bridge_movie_company", "context_movies_ai"
]
for nome in tabelas_gold:
    tabela = f"gold.{nome}"
    df = spark.table(tabela)
    mostrar_contagem(df, tabela)

assert bridge_movie_genre.join(dim_movies.select("sk_movie_id"), "sk_movie_id", "left_anti").limit(1).count() == 0
assert bridge_movie_genre.join(dim_genres.select("sk_genre_id"), "sk_genre_id", "left_anti").limit(1).count() == 0
assert bridge_movie_person.join(dim_people.select("sk_person_id"), "sk_person_id", "left_anti").limit(1).count() == 0
assert bridge_movie_company.join(dim_companies.select("sk_company_id"), "sk_company_id", "left_anti").limit(1).count() == 0
